In [7]:
import numpy as np
import pandas as pd
from sklearn.preprocessing import MinMaxScaler



In [8]:
# 1. Chargement et nettoyage
df = pd.read_csv(r"C:\Users\Roua\Desktop\DataVIZ\IES-Dataviz\Data\donnees_fusionnees_communes.csv", sep=";", encoding="utf-8-sig")
df.columns = df.columns.str.strip()


In [9]:

# 2. Préparation de TOUTES les features orientées vers le "BESOIN"
need = pd.DataFrame(index=df.index)

# --- A. Déficit d'offre médicale (plus l'APL est basse, plus le besoin est élevé) ---
need["deficit_apl_global"] = -df["APL aux médecins généralistes"]
need["deficit_apl_65m"]    = -df["APL aux médecins généralistes de 65 ans ou moins"]
need["deficit_apl_62m"]    = -df["APL aux médecins généralistes de 62 ans ou moins"]
need["deficit_apl_60m"]    = -df["APL aux médecins généralistes de 60 ans ou moins"]

# --- B. Manque de structures existantes (0 structure = besoin max) ---
need["manque_maison_sante"] = -df["nb_maisons_sante"]
need["manque_maison_garde"] = -df["nb_maisons_medicales_garde"]

# --- C. Taille du bassin de population / viabilité du projet ---
need["log_pop_totale"] = np.log1p(df["Population totale 2022"])
need["log_p22_pop"]    = np.log1p(df["P22_POP"])
need["log_pop_std"]    = np.log1p(df["Population standardisée 2022 pour la médecine générale"])

# --- D. Intensité du besoin démographique (toutes les tranches d'âge + mortalité) ---
pop_ref = df["P22_POP"].replace(0, np.nan)
pop_tot_ref = df["Population totale 2022"].replace(0, np.nan)

need["part_0014"] = df["P22_POP0014"] / pop_ref  # Besoin pédiatrique / familles
need["part_1529"] = df["P22_POP1529"] / pop_ref
need["part_3044"] = df["P22_POP3044"] / pop_ref
need["part_4559"] = df["P22_POP4559"] / pop_ref
need["part_6074"] = df["P22_POP6074"] / pop_ref  # Besoin suivi chronique
need["part_75p"]  = df["P22_POP75P"]  / pop_ref  # Très fort besoin médical

need["indice_surbesoin_std"] = df["Population standardisée 2022 pour la médecine générale"] / pop_tot_ref
need["taux_mortalite"]       = df["deces_2022"] / pop_ref

need = need.fillna(need.median())



In [10]:
# 3. Pondération des critères de besoin
poids = {
    # Offre médicale manquante (40% environ du poids total)
    "deficit_apl_global": 2.0,
    "deficit_apl_65m":    1.5,
    "deficit_apl_62m":    1.5,
    "deficit_apl_60m":    2.5,  # Très important : anticipe les départs à la retraite

    # Absence de structures sur place
    "manque_maison_sante": 2.5,
    "manque_maison_garde": 1.5,

    # Masse critique de population à soigner
    "log_pop_totale": 1.0,
    "log_p22_pop":    1.0,
    "log_pop_std":    1.2,

    # Structure de la demande (seniors & enfants pèsent plus lourd)
    "part_0014": 0.6,
    "part_1529": 0.1,
    "part_3044": 0.3,
    "part_4559": 0.5,
    "part_6074": 1.0,
    "part_75p":  1.5,

    # Fragilité sanitaire
    "indice_surbesoin_std": 1.0,
    "taux_mortalite":       0.8,
}

# Normalisation robuste (écrêtage 1% - 99% pour éviter qu'un EHPAD ou une île fausse l'échelle)
need_clipped = need.clip(lower=need.quantile(0.01), upper=need.quantile(0.99), axis=1)
X_norm = pd.DataFrame(
    MinMaxScaler().fit_transform(need_clipped),
    columns=need.columns,
    index=need.index
)

# Calcul du score final sur 100
somme_poids = sum(poids.values())
score = sum(X_norm[col] * w for col, w in poids.items()) / somme_poids * 100
df["score_besoin_maison_medicale"] = score.round(2)


In [11]:

# 4. Règle de décision claire (OUI / À ÉTUDIER / NON)
# Rappel : en France, une APL < 2.5 est le seuil classique de zone sous-dense.
def classer_besoin(row):
    s = row["score_besoin_maison_medicale"]
    aucune_structure = (row["nb_maisons_sante"] == 0) and (row["nb_maisons_medicales_garde"] == 0)
    if s >= 65 and aucune_structure:
        return "OUI - Besoin prioritaire"
    elif s >= 55 or (row["APL aux médecins généralistes"] < 2.5 and aucune_structure):
        return "OUI / À ÉTUDIER - Tension médicale"
    else:
        return "NON - Besoin faible ou déjà couverte"

df["decision_maison_medicale"] = df.apply(classer_besoin, axis=1)

# Indicateurs explicatifs (pour comprendre POURQUOI la commune a besoin ou non)
df["zone_sous_dense_APL_2_5"] = df["APL aux médecins généralistes"] < 2.5
df["chute_apl_retraite_60m"] = (
    df["APL aux médecins généralistes"] - df["APL aux médecins généralistes de 60 ans ou moins"]
).round(3)

# 5. Tri et export
df_res = df.sort_values("score_besoin_maison_medicale", ascending=False)

cols_out = [
    "COM", "departement", "Commune", "Population totale 2022",
    "APL aux médecins généralistes", "APL aux médecins généralistes de 60 ans ou moins",
    "nb_maisons_sante", "nb_maisons_medicales_garde",
    "score_besoin_maison_medicale", "decision_maison_medicale"
]

print("=== TOP 20 DES COMMUNES AYANT LE PLUS BESOIN D'UNE MAISON MÉDICALE ===")
print(df_res[cols_out].head(20).to_string(index=False))

print("\n=== RÉPARTITION DES DÉCISIONS ===")
print(df_res["decision_maison_medicale"].value_counts())

df_res.to_csv("diagnostic_besoin_maison_medicale.csv", sep=";", index=False, encoding="utf-8-sig")

=== TOP 20 DES COMMUNES AYANT LE PLUS BESOIN D'UNE MAISON MÉDICALE ===
  COM       departement                       Commune  Population totale 2022  APL aux médecins généralistes  APL aux médecins généralistes de 60 ans ou moins  nb_maisons_sante  nb_maisons_medicales_garde  score_besoin_maison_medicale decision_maison_medicale
24080          Dordogne                       Capdrot                   461.0                          0.637                                             0.346                 0                           0                         75.52 OUI - Besoin prioritaire
23165            Creuse          Royère-de-Vassivière                   572.0                          1.469                                             0.258                 0                           0                         73.68 OUI - Besoin prioritaire
17484 Charente-Maritime              Port-des-Barques                  1754.0                          1.065                                         